# 🎛️ Aula 16 — Controle Avançado: PID vs MPC

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Tema:** Do PID ao Controle Preditivo (MPC) — modelos de IA em malhas de controle

---

## Contexto

Na Aula 15 otimizamos um ponto de operação. Mas **e se as condições mudarem**? Precisamos de controle em malha fechada — e o MPC (Model Predictive Control) planeja a trajetória futura.

## PID vs MPC

- **PID:** reage ao erro passado (retrovisor)
- **MPC:** planeja a trajetória futura (para-brisa) — otimização em tempo real

**MPC:** a cada instante (1) modelo prediz $\hat{y}_{k+1}...\hat{y}_{k+N}$; (2) otimizador calcula $M$ ações que minimizam o erro futuro; (3) aplica **só a primeira**; (4) repete.

## 3.1 — Simular PID vs MPC

Simule um CSTR com modelo FOPDT e compare os dois controladores.

### Passo 1: Definir o modelo FOPDT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Modelo FOPDT: tau*dy/dt + y = Kp*u
Kp, tau, theta = 0.1, 10.0, 3.0
dt = 1.0   # min

def fopdt(y_prev, u_prev):
    """Um passo de integração (Euler) do FOPDT."""
    return y_prev + (-y_prev + Kp*u_prev) / tau * dt

### Passo 2: Função de simulação (malha fechada)

In [ ]:
def simulate(controller, setpoint, T=120, N=10):
    """Simula PID ou MPC, retorna tempo, y e u."""
    n = int(T/dt)
    y = np.full(n, Kp*5.0)   # estado inicial y=0.5
    u = np.full(n, 5.0)      # entrada inicial
    int_err, prev_err = 0.0, 0.0
    for k in range(1, n):
        ud = u[max(0, k-int(theta/dt))]   # atraso de transporte
        y[k] = fopdt(y[k-1], ud)
        err = setpoint - y[k]
        if controller == 'PID':
            Kp_c, Ki_c, Kd_c = 18.0, 2.5, 0.0
            int_err += err*dt
            der = (err-prev_err)/dt
            u[k] = 5.0 + (Kp_c/Kp)*err + (Ki_c/Kp)*int_err + (Kd_c/Kp)*der
            prev_err = err
        elif controller == 'MPC':
            def cost(us):
                yp = y[k]
                uf = np.concatenate([[u[k-1]], us])
                total = 0.0
                for i in range(N):
                    yp = fopdt(yp, uf[i])
                    total += (setpoint - yp)**2
                total += 0.02*np.sum(np.diff(uf)**2)  # penalidade de esforço
                return total
            res = minimize(cost, np.ones(N)*u[k-1], method='SLSQP',
                           bounds=[(0,10)]*N, options={'maxiter':80,'ftol':1e-6})
            u[k] = res.x[0]
        u[k] = np.clip(u[k], 0, 10)
    return np.arange(n), y, u

### Passo 3: Simular mudança de setpoint (50% → 60%)

In [ ]:
def metricas(y, setpoint, u):
    overshoot = (y.max()-setpoint)/setpoint*100
    band = 0.02*setpoint
    for i in range(15, len(y)):
        if np.all(np.abs(y[i:]-setpoint) < band):
            return overshoot, i, np.sum(np.diff(u)**2)
    return overshoot, len(y), np.sum(np.diff(u)**2)

t_pid, y_pid, u_pid = simulate('PID', 0.60)
t_mpc, y_mpc, u_mpc = simulate('MPC', 0.60)

print(f"{'Controlador':<6} {'Overshoot':>10} {'Estab.':>6} {'Esforço':>8}")
print('-'*36)
for nome, yv, uv in [('PID', y_pid, u_pid), ('MPC', y_mpc, u_mpc)]:
    ov, st, ef = metricas(yv, 0.60, uv)
    print(f"{nome:<6} {ov:>9.1f}% {st:>5d}min {ef:>8.1f}")

### Passo 4: Plotar resposta

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.axhline(0.60, color='gray', linestyle='--', label='Setpoint')
ax1.plot(t_pid, y_pid, label='PID', linewidth=2)
ax1.plot(t_mpc, y_mpc, '--', label=f'MPC (N=10)', linewidth=2)
ax1.set_ylabel('CA (mol/L)'); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title('Resposta a mudança de setpoint — PID vs MPC')

ax2.plot(t_pid, u_pid, label='u_PID')
ax2.plot(t_mpc, u_mpc, '--', label='u_MPC')
ax2.set_ylabel('Entrada u'); ax2.set_xlabel('Tempo (min)')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### ✏️ Pausa reflexiva (2 min)

O PID com sintonia perfeita chega perto do MPC? Em quais métricas?

> _Escreva aqui..._

## 3.2 — Exercício em Grupo: Horizonte de Predição N

Cada grupo testa um horizonte N diferente e aplica uma **perturbação degrau** na alimentação em t=50 min.

| Grupo | N (horizonte) | M (ações) |
|-------|---------------|-----------|
| **A** | 5 | 2 |
| **B** | 10 | 3 |
| **C** | 20 | 5 |
| **D** | 10 | 5 |

In [ ]:
N = 10   # ← mude para o do seu grupo

# Perturbação: redução do ganho em t=50 (simula entupimento/sujeira)
Kp_pert = 0.08
n = 120
y = np.full(n, 0.60); u = np.full(n, 6.0)
for k in range(1, n):
    kk = Kp if k < 50 else Kp_pert
    ud = u[max(0, k-int(theta/dt))]
    y[k] = y[k-1] + (-y[k-1] + kk*ud)/tau*dt
    def cost(us):
        yp = y[k]; uf = np.concatenate([[u[k-1]], us]); total = 0
        kk2 = Kp if k < 50 else Kp_pert
        for i in range(N):
            yp = yp + (-yp + kk2*uf[i])/tau*dt
            total += (0.60-yp)**2
        return total + 0.02*np.sum(np.diff(uf)**2)
    res = minimize(cost, np.ones(N)*u[k-1], method='SLSQP', bounds=[(0,10)]*N, options={'maxiter':60})
    u[k] = res.x[0]

plt.figure(figsize=(10,4))
plt.axvline(50, color='gray', ls='--', label='Perturbação')
plt.plot(np.arange(n), y, linewidth=2)
plt.axhline(0.60, color='gray', ls=':')
plt.xlabel('Tempo (min)'); plt.ylabel('CA'); plt.legend(); plt.grid(alpha=0.3)
plt.title(f'Rejeição de perturbação — MPC com N={N}')
plt.tight_layout(); plt.show()
print(f"CA final: {y[-1]:.3f} (setpoint 0.60) — recuperou após perturbação?")

> **Perguntas:**
> 1. Qual horizonte rejeita a perturbação mais rápido?
> 2. Qual custa menos esforço de controle?
> 3. Horizonte maior = resposta mais suave mas mais lenta?

### 🧠 Desafio extra (NT)

O MPC 'enxerga' 30 min à frente. Se o modelo de predição erra em 10%, o MPC ainda acerta o set-point? O que acontece com a estabilidade?

> A realimentação corrige a cada passo; o horizonte deslizante redireciona. MPC com modelo imperfeito ainda funciona.

> _Escreva sua análise aqui..._

---

## Checklist

- [ ] PID sintonizado (zeigler-nichols)
- [ ] MPC conceitual implementado (scipy.optimize)
- [ ] Horizonte N testado
- [ ] Overshoot medido
- [ ] Esforço de controle comparado
- [ ] Conclusão escrita